Dummy classifier
Create a dummy classifier so that there is a baseline against which to compare to.
Make the loop which will be reused by all the diff condition combinations later.

In [8]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier

import kagglehub
from pathlib import Path
from PIL import Image

%load_ext autoreload
%autoreload 2
import evaluation as ev

/Users/mds/Documents/MSc/10_Project/product-sales/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fold_labels = pd.read_csv('data/fold_labels.csv')

print(fold_labels.shape)
print(fold_labels.head())
print(fold_labels.columns)
print(fold_labels.groupby(['fold', 'split']).size())

(20341, 5)
   external_code  fold  split  label  in_buffer
0              2     0  train      2      False
1              3     0  train      2      False
2              4     0  train      1      False
3              5     0  train      2      False
4              6     0  train      2      False
Index(['external_code', 'fold', 'split', 'label', 'in_buffer'], dtype='object')
fold  split
0     train    3053
      val      1016
1     train    3048
      val      1016
2     train    3051
      val      1016
3     train    3056
      val      1016
4     train    3053
      val      1016
dtype: int64


In [3]:
print(fold_labels[fold_labels.split == 'val'].groupby(['fold', 'in_buffer']).size())
print(fold_labels[fold_labels.split == 'val'].groupby(['fold', 'label']).size())

fold  in_buffer
0     False        769
      True         247
1     False        753
      True         263
2     False        755
      True         261
3     False        772
      True         244
4     False        764
      True         252
dtype: int64
fold  label
0     0        253
      1        509
      2        254
1     0        252
      1        509
      2        255
2     0        256
      1        506
      2        254
3     0        256
      1        507
      2        253
4     0        253
      1        509
      2        254
dtype: int64


In [ ]:
# dummy classifier test

training_rows   = fold_labels[(fold_labels.fold == 0) & (fold_labels.split == 'train')]
validation_rows = fold_labels[(fold_labels.fold == 0) & (fold_labels.split == 'val')]

print(f"Training rows: {len(training_rows)}")
print(f"Validation rows: {len(validation_rows)}")

# fill the X with 0s as a placeholder for the dummy
# same length as the training rows
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(np.zeros((len(training_rows), 1)), training_rows['label'])
predictions = dummy.predict(np.zeros((len(validation_rows), 1)))

label_counts = training_rows['label'].value_counts()

# should return one unique value as it's predicting most frequent
print(f"Unique predictions: {np.unique(predictions)}")
print(f"Training labels: \n{label_counts}")

Training rows: 3053
Validation rows: 1016
Unique predictions: [1]
Training labels: 
label
1    1021
2    1016
0    1016
Name: count, dtype: int64


In [14]:
# balanced accuracy should be 0.3333 
# non_buffer should be < full as it's only selecting the "extremes"
result = ev.score_fold(validation_rows['label'], predictions, validation_rows['in_buffer'])

print(pd.Series(result))

balanced_accuracy_full            0.333333
balanced_accuracy_nonbuffer       0.333333
n_full                         1016.000000
n_nonbuffer                     769.000000
dtype: float64


In [ ]:
# loop for training folds

results = []

for fold in range(5):
    training_rows   = fold_labels[(fold_labels.fold == fold) & (fold_labels.split == 'train')]
    validation_rows = fold_labels[(fold_labels.fold == fold) & (fold_labels.split == 'val')]

    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(np.zeros((len(training_rows), 1)), training_rows['label'])
    predictions = dummy.predict(np.zeros((len(validation_rows), 1)))

    fold_result = ev.score_fold(validation_rows['label'], predictions, validation_rows['in_buffer'])

    results.append({'fold': fold, **fold_result})

pd.DataFrame(results)

,fold,balanced_accuracy_full,balanced_accuracy_nonbuffer,n_full,n_nonbuffer
0,0,0.333333,0.333333,1016,769
1,1,0.333333,0.333333,1016,753
2,2,0.333333,0.333333,1016,755
3,3,0.333333,0.333333,1016,772
4,4,0.333333,0.333333,1016,764
